In [1]:
# Import the libraries, connect to the ENTSO-E API and create the folder for the raw files
import os
import pandas as pd
from entsoe import EntsoePandasClient

token = open("../entsoe_token.txt").read().strip()
client = EntsoePandasClient(api_key=token)
os.makedirs("../data/raw/entsoe", exist_ok=True)

In [2]:
# Bidding zones to download (one per country) and the period
zones = {
    "PT": "Portugal", "ES": "Spain", "FR": "France", "BE": "Belgium", "NL": "Netherlands",
    "DE_LU": "Germany", "AT": "Austria", "CH": "Switzerland", "IT_NORD": "Italy (North)",
    "PL": "Poland", "CZ": "Czechia", "SK": "Slovakia", "HU": "Hungary", "SI": "Slovenia",
    "HR": "Croatia", "RO": "Romania", "BG": "Bulgaria", "GR": "Greece", "DK_1": "Denmark (West)",
    "SE_3": "Sweden (Stockholm)", "NO_1": "Norway (Oslo)", "FI": "Finland", "EE": "Estonia",
    "LV": "Latvia", "LT": "Lithuania", "IE_SEM": "Ireland",
}
start = pd.Timestamp("2019-01-01", tz="Europe/Brussels")
today = pd.Timestamp.today(tz="Europe/Brussels").normalize()
end_prices = today + pd.Timedelta(days=2)
end_generation = today

In [3]:
# Download the hourly prices of each zone since 2019 (zones already downloaded are skipped)
for zone in zones:
    file = "../data/raw/entsoe/prices_" + zone + ".csv"
    if os.path.exists(file):
        print(zone, "already downloaded")
        continue
    try:
        prices = client.query_day_ahead_prices(zone, start=start, end=end_prices)
        prices = prices.resample("h").mean().tz_convert("UTC")
        prices.to_csv(file, index_label="time_utc", header=["price"])
        print(zone, len(prices), "hours")
    except Exception as error:
        print(zone, "failed:", error)

PT 68111 hours
ES 68111 hours
FR 68087 hours
BE 68111 hours
NL 68087 hours
DE_LU 68087 hours
AT 68111 hours
CH 68111 hours
IT_NORD 68087 hours
PL 68111 hours
CZ 68111 hours
SK 68111 hours
HU 68111 hours
SI 68111 hours
HR 68111 hours
RO 68111 hours
BG 68111 hours
GR 68111 hours
DK_1 68111 hours
SE_3 68111 hours
NO_1 68111 hours
FI 68111 hours
EE 68111 hours
LV 68111 hours
LT 68111 hours
IE_SEM 67895 hours


In [4]:
# Check: average price in 2025 in each zone
for zone in zones:
    file = "../data/raw/entsoe/prices_" + zone + ".csv"
    if os.path.exists(file):
        data = pd.read_csv(file)
        year_2025 = data[data["time_utc"].str[:4] == "2025"]
        print(zones[zone], round(year_2025["price"].mean(), 1), "€/MWh")

Portugal 66.2 €/MWh
Spain 65.3 €/MWh
France 61.1 €/MWh
Belgium 82.6 €/MWh
Netherlands 86.8 €/MWh
Germany 89.3 €/MWh
Austria 98.9 €/MWh
Switzerland 101.7 €/MWh
Italy (North) 115.9 €/MWh
Poland 104.3 €/MWh
Czechia 96.8 €/MWh
Slovakia 102.7 €/MWh
Hungary 108.5 €/MWh
Slovenia 104.6 €/MWh
Croatia 105.4 €/MWh
Romania 108.2 €/MWh
Bulgaria 106.9 €/MWh
Greece 103.6 €/MWh
Denmark (West) 81.1 €/MWh
Sweden (Stockholm) 46.2 €/MWh
Norway (Oslo) 58.4 €/MWh
Finland 40.5 €/MWh
Estonia 80.4 €/MWh
Latvia 85.7 €/MWh
Lithuania 85.3 €/MWh
Ireland 114.4 €/MWh


In [5]:
# Download one series (solar, wind or consumption) for one zone, as hourly values in UTC
def download(zone, kind):
    try:
        if kind == "load":
            result = client.query_load(zone, start=start, end=end_generation)
        else:
            result = client.query_generation(zone, start=start, end=end_generation, psr_type=kind)
        if isinstance(result, pd.DataFrame):
            result = result.iloc[:, 0]
        return result.resample("h").mean().tz_convert("UTC")
    except Exception as error:
        print("   ", zone, kind, "not available:", error)
        return None

In [ ]:
# Download solar, wind and consumption of each zone since 2019 (zones already downloaded are skipped)
series_codes = {"solar": "B16", "wind_onshore": "B19", "wind_offshore": "B18", "load": "load"}

for zone in zones:
    file = "../data/raw/entsoe/generation_" + zone + ".csv"
    if os.path.exists(file):
        print(zone, "already downloaded")
        continue
    table = pd.DataFrame()
    for name in series_codes:
        series = download(zone, series_codes[name])
        if series is not None:
            table[name] = series
    table.to_csv(file, index_label="time_utc")
    print(zone, len(table), "hours,", list(table.columns))

PT 68063 hours, ['solar', 'wind_onshore', 'wind_offshore', 'load']
ES 68063 hours, ['solar', 'wind_onshore', 'wind_offshore', 'load']
    FR B18 not available: Can only union MultiIndex with MultiIndex or Index of tuples, try mi.to_flat_index().union(other) instead.
FR 68063 hours, ['solar', 'wind_onshore', 'load']
BE 68063 hours, ['solar', 'wind_onshore', 'wind_offshore', 'load']
